Adattisztítás

In [1]:
import pandas as pd
import numpy as np

In [2]:
dataSet = pd.read_csv('data/lakasok.csv')
dataSet.head()

,terulet_m2,szobak,kerulet,emelet,lift,epitesi_ev,allapot,futes,erkely,nm_ar_ft
0,54,1,III,9.0,igen,1907.0,felújított,hőszivattyú,igen,1168000
1,58,1,XVIII,5.0,igen,1894.0,újszerű,távfűtés,nem,834000
2,55,2,XVI,6.0,igen,1923.0,közepes,gáz cirkó,igen,723000
3,63,2,XI,2.0,igen,1891.0,közepes,gáz cirkó,igen,1195000
4,100,3,X,0.0,nem,1961.0,felújított,távfűtés,igen,588000


In [ ]:
dataSet.isna().sum()

terulet_m2     0
szobak         0
kerulet        0
emelet         9
lift           0
epitesi_ev    12
allapot        0
futes          8
erkely         6
nm_ar_ft       0
dtype: int64

In [ ]:
dataSet.dropna(inplace=True)
dataSet.isna().sum()

terulet_m2    0
szobak        0
kerulet       0
emelet        0
lift          0
epitesi_ev    0
allapot       0
futes         0
erkely        0
nm_ar_ft      0
dtype: int64

In [ ]:
dataSet['terulet_m2'] = dataSet['terulet_m2'].astype(float)
dataSet['szobak'] = dataSet['szobak'].astype(float)
dataSet['nm_ar_ft'] = dataSet['nm_ar_ft'].astype(float)

In [ ]:
print("Lakás állapota: ", dataSet['allapot'].unique())
print("Fűtés: ", dataSet['futes'].unique())

Lakás állapota:  <StringArray>
['felújított', 'újszerű', 'közepes', 'felújítandó']
Length: 4, dtype: str
Fűtés:  <StringArray>
['hőszivattyú', 'távfűtés', 'gáz cirkó', 'elektromos']
Length: 4, dtype: str


Know your encoding:
- Mapping: A kategóriák között van kisebb–nagyobb viszony pl állapot, iskolai végzettség, ruhaméret

- One-hot-encoding: a kategóriák csak különböznek egymástól mint a fűtés, kerület

- Target-encoding: a kategória helyére a hozzá tartozó átlagos célérték kerül (pl. kerület → átlag nm-ár). Csak trainből számolva, különben leakage!

In [ ]:
dataSet['lift'] = dataSet['lift'].map({'igen': 1, 'nem': 0})
dataSet['erkely'] = dataSet['erkely'].map({'igen': 1, 'nem': 0})

dataSet['allapot'] = dataSet['allapot'].map({
    'felújítandó': 0, 
    'közepes': 1,
    'felújított': 2,
    'újszerű': 3
    })

In [ ]:
dataSet = pd.get_dummies(dataSet, columns=['futes'], dtype=int)
dataSet = pd.get_dummies(dataSet, columns=['kerulet'], dtype=int)

In [ ]:
dataSet.head()

,terulet_m2,szobak,emelet,lift,epitesi_ev,allapot,erkely,nm_ar_ft,futes_elektromos,futes_gáz cirkó,...,kerulet_XIV,kerulet_XIX,kerulet_XV,kerulet_XVI,kerulet_XVII,kerulet_XVIII,kerulet_XX,kerulet_XXI,kerulet_XXII,kerulet_XXIII
0,54.0,1.0,9.0,1,1907.0,2,1,1168000.0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,58.0,1.0,5.0,1,1894.0,3,0,834000.0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,55.0,2.0,6.0,1,1923.0,1,1,723000.0,0,1,...,0,0,0,1,0,0,0,0,0,0
3,63.0,2.0,2.0,1,1891.0,1,1,1195000.0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,100.0,3.0,0.0,0,1961.0,2,1,588000.0,0,0,...,0,0,0,0,0,0,0,0,0,0


Know your split:
```
Teljes adat
├── TRAIN 70%  → ezen tanul a háló
└── TEMP  30%
    ├── VAL  15%  → tanítás közben figyeljük (túltanul-e, mikor álljon le)
    └── TEST 15%  → csak a legvégén, egyszer mérünk rajta
```
- train_test_split visszatérés: nagy rész, levágott rész (`X_a, X_b, y_a, y_b`), a test_size mindig a bemenet arányára vonatkozik

In [ ]:
# y normalizálása még a szétosztás előtt
y = dataSet['nm_ar_ft'] / 1000000

In [11]:
from sklearn.model_selection import train_test_split

X = dataSet.drop('nm_ar_ft', axis=1)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

(395, 34) (85, 34) (85, 34)
(395,) (85,) (85,)


Normalizálás
$$x_{\text{norm}} = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}$$

- $x$: az eredeti, átalakítandó érték
- $x_{\text{min}}$: az adathalmaz (vagy oszlop) legkisebb értéke
- $x_{\text{max}}$: az adathalmaz (vagy oszlop) legnagyobb értéke
- $x_{\text{norm}}$: a normalizált érték, ami alapértelmezés szerint a $[0, 1]$ intervallumra esik.

In [ ]:
nonBinaryCols = ["terulet_m2", "szobak", "emelet", "epitesi_ev", "allapot"]

# Dictionary<string, (double Min, double Max)>
scaleParams = {} # TODO: ez majd később fog kelleni amikor új adatokat akarunk majd normalizálni, hogy ugyanazt a skálát használjuk

for col in nonBinaryCols:
    colMin = X_train[col].min()
    colMax = X_train[col].max()
    scaleParams[col] = (colMin, colMax)

    X_train[col] = (X_train[col] - colMin) / (colMax - colMin)
    X_val[col] = (X_val[col] - colMin) / (colMax - colMin)
    X_test[col] = (X_test[col] - colMin) / (colMax - colMin)

In [13]:
print("TRAIN")
X_train.head()

TRAIN


,terulet_m2,szobak,emelet,lift,epitesi_ev,allapot,erkely,futes_elektromos,futes_gáz cirkó,futes_hőszivattyú,...,kerulet_XIV,kerulet_XIX,kerulet_XV,kerulet_XVI,kerulet_XVII,kerulet_XVIII,kerulet_XX,kerulet_XXI,kerulet_XXII,kerulet_XXIII
147,0.215517,0.25,0.9,1,0.962963,1.000000,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
317,0.422414,0.25,0.7,0,0.592593,0.666667,1,0,1,0,...,0,0,0,0,1,0,0,0,0,0
448,0.474138,0.50,0.3,1,0.570370,0.333333,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
591,0.301724,0.50,0.9,1,0.037037,0.333333,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
20,0.431034,0.50,0.8,1,0.022222,0.000000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
print("VALIDATION")
X_val.head()

VALIDATION


,terulet_m2,szobak,emelet,lift,epitesi_ev,allapot,erkely,futes_elektromos,futes_gáz cirkó,futes_hőszivattyú,...,kerulet_XIV,kerulet_XIX,kerulet_XV,kerulet_XVI,kerulet_XVII,kerulet_XVIII,kerulet_XX,kerulet_XXI,kerulet_XXII,kerulet_XXIII
291,0.379310,0.5,0.4,1,0.696296,0.333333,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
261,0.448276,0.5,0.0,0,0.022222,0.333333,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
287,0.103448,0.0,0.4,1,0.637037,1.000000,1,0,1,0,...,1,0,0,0,0,0,0,0,0,0
412,0.560345,0.5,0.6,1,0.296296,0.000000,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
34,0.568966,0.5,0.4,0,0.962963,1.000000,1,0,0,1,...,0,0,0,0,0,0,0,1,0,0


In [15]:
print("TEST")
X_test.head()

TEST


,terulet_m2,szobak,emelet,lift,epitesi_ev,allapot,erkely,futes_elektromos,futes_gáz cirkó,futes_hőszivattyú,...,kerulet_XIV,kerulet_XIX,kerulet_XV,kerulet_XVI,kerulet_XVII,kerulet_XVIII,kerulet_XX,kerulet_XXI,kerulet_XXII,kerulet_XXIII
47,0.129310,0.25,0.1,1,0.007407,0.333333,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
521,0.353448,0.50,0.8,1,0.303704,0.000000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
308,0.224138,0.00,0.0,0,0.362963,0.000000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
263,0.422414,0.50,0.8,1,0.170370,0.666667,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
598,0.594828,0.75,0.2,1,0.022222,1.000000,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0


# Neurális háló: Pytorch
- pip install torch -> CPU only, a GPU használatához ki kell keresni a jó verziót az oldalukon

In [16]:
# ez könnyen eldönti hogy kell-e a GPU-s verzióval bajlódni
import torch
print(torch.cuda.is_available())

False


### Know your data types
1
A PyTorch hálója csak **tensort** fogad el, DataFrame-et nem, ezért konvertálni kell.

- **DataFrame:** táblázat fejléccel, indexszel, oszloponként eltérő típussal (`DataTable`)
- **Numpy tömb:** sima számtömb, egy típus, nincs fejléc (`float[,]`)
- **Tensor:** számtömb, ami **gradienst tud számolni** és GPU-ra rakható (ha a GPU-n van CUDA). Csak numpy-ból lehet konvertálni, DataFrame-ből nem.

```
DataFrame → numpy (float32) → tensor
```

**Csapdák:**

- `float32` kell, mert a pandas alapból `float64`-et ad, és ez hibát okoz a háló súlyaival
- `y` alakja `(N, 1)` legyen, ne `(N,)`, különben a loss rosszul számol

In [17]:
X_train_np = X_train.to_numpy(dtype="float32")
X_train_t = torch.tensor(X_train_np)
y_train_np = y_train.to_numpy(dtype="float32").reshape(-1, 1)
y_train_t = torch.tensor(y_train_np)
print("TRAIN")
print("X train:", X_train_t.shape, X_train_t.dtype)
print("y train:", y_train_t.shape, y_train_t.dtype)

X_val_np = X_val.to_numpy(dtype="float32")
X_val_t = torch.tensor(X_val_np)
y_val_np = y_val.to_numpy(dtype="float32").reshape(-1, 1)
y_val_t = torch.tensor(y_val_np)
print("VALIDATION")
print("X validation:", X_val_t.shape, X_val_t.dtype)
print("y validation:", y_val_t.shape, y_val_t.dtype)

X_test_np = X_test.to_numpy(dtype="float32")
X_test_t = torch.tensor(X_test_np)
y_test_np = y_test.to_numpy(dtype="float32").reshape(-1, 1)
y_test_t = torch.tensor(y_test_np)
print("TEST")
print("X test:", X_test_t.shape, X_test_t.dtype)
print("y test:", y_test_t.shape, y_test_t.dtype)

TRAIN
X train: torch.Size([395, 34]) torch.float32
y train: torch.Size([395, 1]) torch.float32
VALIDATION
X validation: torch.Size([85, 34]) torch.float32
y validation: torch.Size([85, 1]) torch.float32
TEST
X test: torch.Size([85, 34]) torch.float32
y test: torch.Size([85, 1]) torch.float32


#### Háló felépítése

### HousePriceNet felépítése

**nn.Linear(bemenetek, neuronok)** – mind a 16 neuron megkapja mind a 34 oszlopot, súlyozott összeget számol + bias

**Aktivációs függvény** – a réteg kimenetére hívjuk. Ha 0-t ad, az a neuron **az adott bemenetre** nem kapcsol be.

- `nn.ReLU()` – negatív → 0. **Alapválasztás rejtett rétegbe.**
- `nn.LeakyReLU(0.01)` – negatívat nem nullázza, csak kicsinyíti
- `nn.Sigmoid()` – [0; 1] közé nyom
- `nn.Tanh()` – [-1; 1] közé nyom

**self.output = nn.Linear(16, 1)** – a 16 neuron kimenetéből súlyozott összeg + bias → egy jósolt ár.
A kimeneten:
- nincs aktiváció: regresszió, bármilyen szám lehet
- Sigmoid aktiváció: logisztikus regresszió, két osztály van, vagy igen vagy nem

**forward** – a műveletek sorrendje, bármilyen bemenetre (train, val, test, új lakás):

1. `x = self.hidden(x)` → `[N, 34]` → `[N, 16]`
2. `x = self.activation(x)` → ReLU az egész rétegre
3. `x = self.output(x)` → `[N, 16]` → `[N, 1]`

### Know your activation function

Tanításkor a hiba visszafelé halad a rétegeken, és minden rétegnél megszorzódik az aktiváció deriváltjával.
Ha ez a derivált kicsi, a hiba rétegről rétegre elhal így a háló eleje nem tanul (**vanishing gradient**).

**ReLU** – max(0, x)
- Pozitív oldalon a derivált mindig 1 ezzel a hiba nem halványul és a mély hálók is tanulnak
- Ha egy neuron minden bemenetre negatívat ad, a derivált 0, és többé nem tanul (**dying ReLU**)
- A rejtett rétegeknél szinte mindig ez az alapválasztás

**LeakyReLU** – A negatív oldalon: 0.01 * x
- A negatív oldalon is van kis derivált → a neuron nem tud végleg „meghalni”
- Ugyan úgy a rejtett rétegeknél hasznos, ha a ReLU-nál sok a halott neuron

**Sigmoid** – [0; 1]
- Mindkét végén ellaposodik, a derivált max. 0.25 így a mély hálóban elhal a gradiens
- A kimenete valószínűségként vagy arányként használható
- Ha a kimeneti réteg két osztály akkor hasznos (pl. beteg/nem beteg)
- LSTM kapuk: mennyit engedjen át: 0 = semmit, 1 = mindent

**Tanh** – [-1; 1]
- Szintén ellaposodik, de 0 körül szimmetrikus, talán jobb, mint a Sigmoid
- LSTM/RNN-ben értelmes ha a cél [-1; 1] közötti

In [ ]:
import torch.nn as nn

class HousePriceNet(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.hidden(x)
        x = self.activation(x)
        x = self.output(x)
        return x

In [20]:
model = HousePriceNet(34, 16)
print(model)
print(model(X_train_t[:5]))

HousePriceNet(
  (hidden): Linear(in_features=34, out_features=16, bias=True)
  (activation): ReLU()
  (output): Linear(in_features=16, out_features=1, bias=True)
)
tensor([[-0.4292],
        [-0.3916],
        [-0.4406],
        [-0.4296],
        [-0.4187]], grad_fn=<AddmmBackward0>)


Hibafüggvények: A hibafüggvények hiba érétékének minimalizálásval tanúl a háló, minnél kisebb annál pontosabb a háló, viszont ez csak pozitív hiba értékeknél működik
- nn.MSELoss(): Négyzetesen számol hibát
- nn.MAELoss(): Abszolút értékkel számol hibát

Optimalizáló: A hibafüggvény alapján elmozdítja az összes súlyt egy kicsit abba az irányba, amerre a hiba csökken.
- model.parameters(): átadja neki a háló összes súlyát
- lr (learning rate): Mekkorát lépjen egy lépéssel.
    - Ha túl nagy, átugorja a jó megoldást, és a loss ugrál vagy elszáll.
    - Ha túl kicsi, a tanítás nagyon lassú.

### Know your optimizer function

**Mindegyik ugyanazt csinálja:** A hibafüggvény alapján elmozdítja az összes súlyt a jó irányba. 
$$súly = súly - lépésméret × gradiens$$
**A különbség:** hogyan, mennyire intelligensen számolják a lépésméretet.

**SGD** – `torch.optim.SGD(params, lr=0.01)`
- Minden súly ugyanakkora lépést tesz, csak a gradiens irányába
- Gond: lapos részeken lassú, szűk völgyekben cikázik
- **Hol:** ez... iagazából egy szar de könnyű megérteni az optimalizálás lényegét szóval többnyire csak tanulásnál van értelme

**SGD + momentum** – `torch.optim.SGD(params, lr=0.01, momentum=0.9)`
- Megjegyzi az előző lépések irányát
- Kevesebb cikázás, gyorsabban halad a lejtőn
- **Hol:** képes hálók (CNN), ha van időd az lr-t hangolni. Gyakran jobban általánosít, mint az Adam.

**RMSprop** – `torch.optim.RMSprop(params, lr=0.001)`
- Minden súly saját lépésméretet kap: ahol eddig nagy volt a gradiens, ott kisebbet lép
- **Hol:** megerősítéses tanulás

**Adam** – `torch.optim.Adam(params, lr=0.001)`
- Momentum + RMSprop egyben: lendület ÉS súlyonkénti lépésméret
- Kevés hangolással is jól megy, a 0.001 szinte mindig működik kezdésnek
- **Hol:** ez az alapválasztás, ha nem tudod, mit válassz...

**AdamW** – `torch.optim.AdamW(params, lr=0.001, weight_decay=0.01)`
- Adam + helyesen működő súlycsökkentés a túl nagy súlyokat bünteti ezzel kisebb a túltanulás
- **Hol:** ha túltanul a háló vagy LLM (ott ez standard)

In [21]:
lossFunction = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)